# Analysis 3 · URL path length & structure

Analysis 3 (a–g): **length of URL paths per domain/year of directory paths
only, filenames excluded.** Uses the same corrected filename rule as
`a2_paths.ipynb` (a trailing `.<2–4>` is a filename only if it contains a
**letter**, so numeric endings like `.2345` stay in the path).

Deduplicated to one unique URL per `(crawl_year, surtkey)`, `dns:` removed.
Outputs → `analysis/deliverables/`. Companions: `a1_subdomains.ipynb`, `a2_paths.ipynb`.

> Runtime note: several cells re-scan each DB; expect a few minutes across all 15.

In [ ]:
import sys, os
os.umask(0o000)
sys.path.insert(0, os.path.abspath('..')); sys.path.insert(0, os.path.abspath('.'))
import config
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']

def _writable_dir(preferred='deliverables'):
    for cand in (preferred, os.path.expanduser('~/eot_deliverables')):
        try:
            os.makedirs(cand, exist_ok=True)
            try: os.chmod(cand, 0o777)
            except OSError: pass
            t=os.path.join(cand,'.wtest'); open(t,'w').close(); os.remove(t); return cand
        except OSError: continue
    raise RuntimeError('no writable output dir')
OUTDIR=_writable_dir(); print('OUTDIR =', os.path.abspath(OUTDIR))
DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

def per_domain(sql):
    frames=[]
    for dom,path in DBS.items():
        con=duckdb.connect(str(path), read_only=True); df=con.sql(sql).df(); con.close()
        df.insert(0,'domain',dom); frames.append(df)
    out=pd.concat(frames, ignore_index=True)
    if 'crawl_year' in out.columns: out['crawl_year']=out['crawl_year'].astype(str)
    return out

def year_cols(df): return [y for y in YEAR_ORDER if y in df.columns]
def save(name):
    p=os.path.join(OUTDIR,name)
    try:
        if os.path.exists(p): os.remove(p)
    except OSError: pass
    return p

# RFC 3986 section 2.2 reserved chars (per EM's list) + % for pct-encoding, via chr() codes
RESERVED = {'q(?)':63,'hash(#)':35,'lbrack([)':91,'rbrack(])':93,'at(@)':64,
            'bang(!)':33,'dollar($)':36,'amp(&)':38,'apos(apostrophe)':39,'lparen(()':40,'rparen())':41,
            'star(*)':42,'plus(+)':43,'comma(,)':44,'semi(;)':59,'eq(=)':61,'pct(%)':37}

In [ ]:
BASE = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey, min(regexp_extract(surtkey,'\\)([^?]*)',1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1,2),
paths AS (
  SELECT crawl_year,
    CASE WHEN regexp_matches(url_path,'/[^/]+\\.[a-zA-Z0-9]{2,4}$')
          AND regexp_matches(regexp_extract(url_path,'\\.([a-zA-Z0-9]{2,4})$',1),'[a-zA-Z]')
         THEN regexp_replace(url_path,'/[^/]+\\.[a-zA-Z0-9]{2,4}$','') ELSE rtrim(url_path,'/') END AS dir_path
  FROM dedup),
segs AS (
  SELECT crawl_year, dir_path,
    CASE WHEN trim(dir_path,'/')='' THEN 0 ELSE len(string_split(trim(dir_path,'/'),'/')) END AS depth,
    list_extract(string_split(trim(dir_path,'/'),'/'),1) AS seg1,
    list_extract(string_split(trim(dir_path,'/'),'/'),2) AS seg2,
    list_extract(string_split(trim(dir_path,'/'),'/'),3) AS seg3,
    list_extract(string_split(trim(dir_path,'/'),'/'),4) AS seg4,
    list_extract(string_split(trim(dir_path,'/'),'/'),5) AS seg5
  FROM paths),
longseg AS (
  SELECT crawl_year, seg1 AS s FROM segs WHERE seg1 IS NOT NULL
  UNION ALL SELECT crawl_year,seg2 FROM segs WHERE seg2 IS NOT NULL
  UNION ALL SELECT crawl_year,seg3 FROM segs WHERE seg3 IS NOT NULL
  UNION ALL SELECT crawl_year,seg4 FROM segs WHERE seg4 IS NOT NULL
  UNION ALL SELECT crawl_year,seg5 FROM segs WHERE seg5 IS NOT NULL)
'''
print('BASE CTE ready (paths-only; filename requires a letter, so .2345 stays in the path)')

## 3 · path length + (a–e) segment presence

`depth` = number of directory segments. For each domain/year: mean/median/max
depth, and how many URLs **have / don't have** `path_seg1 … path_seg5`.

In [ ]:
LEN_SQL = BASE + '''
SELECT crawl_year, count(*) AS unique_urls,
  round(avg(depth),2) AS mean_depth, median(depth) AS median_depth, max(depth) AS max_depth,
  sum((depth>=1)::int) AS has_seg1, sum((depth>=2)::int) AS has_seg2, sum((depth>=3)::int) AS has_seg3,
  sum((depth>=4)::int) AS has_seg4, sum((depth>=5)::int) AS has_seg5
FROM segs GROUP BY 1 ORDER BY 1'''
plen = per_domain(LEN_SQL)
for i in range(1,6):
    plen[f'lacks_seg{i}'] = plen['unique_urls'] - plen[f'has_seg{i}']
plen.to_csv(save('a3_path_length_and_presence.csv'), index=False)
print('path length + segment presence per domain/year -> a3_path_length_and_presence.csv')
plen.head()

### Median path depth by domain and year (heatmap)

In [ ]:
piv = plen.pivot(index='domain', columns='crawl_year', values='median_depth')
piv = piv[year_cols(piv)].reindex([d for d in config.TARGET_DOMAINS if d in piv.index])
plt.figure(figsize=(9,7))
sns.heatmap(piv, annot=True, fmt='g', cmap='viridis', cbar_kws={'label':'median directory depth'}, linewidths=.5)
plt.title('Median URL path depth (directory segments), by domain and year')
plt.xlabel('crawl year'); plt.ylabel(''); plt.tight_layout()
plt.savefig(save('a3_median_depth.png'), dpi=150, bbox_inches='tight'); plt.show()

### Segment presence — have vs don't-have each position (example domain)

In [ ]:
ex = 'doi.gov' if 'doi.gov' in DBS else list(DBS)[0]
d = plen[plen.domain==ex].set_index('crawl_year')
rows=[]
for i in range(1,6):
    rows.append(pd.DataFrame({'crawl_year':d.index, 'position':f'seg{i}',
        'have':d[f'has_seg{i}'].values, 'dont_have':d[f'lacks_seg{i}'].values,
        'pct_have':(100*d[f'has_seg{i}']/d['unique_urls']).round(1).values}))
pres = pd.concat(rows)
print(f'Segment presence — {ex}')
pres.pivot(index='position', columns='crawl_year', values='pct_have')[year_cols(d.reset_index())]

## 3f · longest segment at each position

The single longest directory segment at positions 1–5, per domain (across years).

In [ ]:
LONGEST_SQL = BASE + '''
SELECT 1 AS pos, arg_max(seg1,length(seg1)) AS longest_segment, max(length(seg1)) AS len FROM segs WHERE seg1 IS NOT NULL
UNION ALL SELECT 2, arg_max(seg2,length(seg2)), max(length(seg2)) FROM segs WHERE seg2 IS NOT NULL
UNION ALL SELECT 3, arg_max(seg3,length(seg3)), max(length(seg3)) FROM segs WHERE seg3 IS NOT NULL
UNION ALL SELECT 4, arg_max(seg4,length(seg4)), max(length(seg4)) FROM segs WHERE seg4 IS NOT NULL
UNION ALL SELECT 5, arg_max(seg5,length(seg5)), max(length(seg5)) FROM segs WHERE seg5 IS NOT NULL
ORDER BY pos'''
longest = per_domain(LONGEST_SQL)
longest.to_csv(save('a3_longest_segment_per_position.csv'), index=False)
print('longest segment per position -> a3_longest_segment_per_position.csv')
longest.head(10)

## 3f · segments using the most word separators (`-` and `_`)

Hyphens/underscores typically separate words, so these are the most "word-rich"
segments — human-readable slugs (vs. machine query blobs). Top 25 per domain.

In [ ]:
SEP_SQL = BASE + '''
SELECT s AS segment,
  length(s)-length(replace(s,'-','')) AS hyphens,
  length(s)-length(replace(s,'_','')) AS underscores
FROM longseg GROUP BY s
ORDER BY (length(s)-length(replace(s,'-','')) + length(s)-length(replace(s,'_',''))) DESC
LIMIT 25'''
seps = per_domain(SEP_SQL)
seps['separators'] = seps['hyphens'] + seps['underscores']
seps = seps.sort_values(['domain','separators'], ascending=[True,False])
seps.to_csv(save('a3_most_separators.csv'), index=False)
print('top word-separator segments -> a3_most_separators.csv')
seps.head(15)

## 3g · reserved characters (RFC 3986 §2.2) in path segments

Counts of path **segments** containing each reserved character — gen-delims
`? # [ ] @`, sub-delims `! $ & ' ( ) * + , ; =`, plus `%` (percent-encoding).
Reserved chars in a path usually signal machine construction / encoding.

In [ ]:
any_seg = " OR ".join(f"contains(s,chr({c}))" for c in RESERVED.values())
cols = ",\n  ".join(f'sum(contains(s,chr({c}))::int) AS "{name}"' for name,c in RESERVED.items())
RES_SQL = BASE + f'''
SELECT crawl_year, count(*) AS total_segments,
  sum(({any_seg})::int) AS any_reserved,
  {cols}
FROM longseg GROUP BY 1 ORDER BY 1'''
reserved = per_domain(RES_SQL)
reserved['pct_any_reserved'] = (100*reserved['any_reserved']/reserved['total_segments']).round(3)
reserved.to_csv(save('a3_reserved_characters.csv'), index=False)

# per-character totals across all domains/years
charcols = [f'"{n}"'.strip('"') for n in RESERVED]
totals = reserved[list(RESERVED.keys())].sum().sort_values(ascending=False).astype(int)
print('Reserved-char occurrences (segments), all domains/years:')
display(totals.to_frame('segments_containing'))
print('\n% of segments with ANY reserved char, by domain/year:')
reserved.pivot(index='domain', columns='crawl_year', values='pct_any_reserved')[year_cols(reserved)]

## 3g · QA — example URLs with reserved characters in the path

`url` + `surtkey` for paths containing a reserved char, for manual review.

In [ ]:
any_dir = " OR ".join(f"contains(dir_path,chr({c}))" for c in RESERVED.values())
SAMPLE_SQL = f'''
WITH dedup AS (
  SELECT crawl_year, surtkey, min(url) AS url, min(regexp_extract(surtkey,'\\)([^?]*)',1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1,2),
paths AS (
  SELECT crawl_year, url, surtkey,
    CASE WHEN regexp_matches(url_path,'/[^/]+\\.[a-zA-Z0-9]{{2,4}}$')
          AND regexp_matches(regexp_extract(url_path,'\\.([a-zA-Z0-9]{{2,4}})$',1),'[a-zA-Z]')
         THEN regexp_replace(url_path,'/[^/]+\\.[a-zA-Z0-9]{{2,4}}$','') ELSE rtrim(url_path,'/') END AS dir_path
  FROM dedup)
SELECT crawl_year, dir_path, url, surtkey FROM paths
WHERE {any_dir}
ORDER BY crawl_year LIMIT 100'''
ressamp = per_domain(SAMPLE_SQL)
ressamp.to_csv(save('a3_reserved_character_examples.csv'), index=False)
print(f"{len(ressamp):,} example URLs with reserved chars -> a3_reserved_character_examples.csv")
ressamp[['domain','crawl_year','url']].head(15)